# Stage 4 — Model 5: Support Vector Classifier (`SVC` with RBF Kernel)
**Project:** 10-Year Coronary Heart Disease (CHD) Risk Prediction (Framingham Heart Study)  
**Author:** Saif Asaad Naeem | Al-Maaqal University — Department of AI Engineering

---
### Notebook Roadmap:
1. **Environment Setup & Data Loading:** Load `framingham_balanced.csv` and apply the unified 80%/20% stratified split.
2. **Stage 1 — Baseline Model Evaluation:** Evaluate the default `SVC (C=1.0, gamma='scale')`.
3. **Stage 2 — Continuous Exploration via `RandomizedSearchCV ()`:** Sample continuous log-uniform distributions (`loguniform(...)`) for $C$ and $\gamma$ using parentheses `()`.
4. **Stage 3 — Exhaustive Exploitation via `GridSearchCV []`:** Pinpoint the optimal RBF kernel boundary ($C=3.905, \gamma=0.6376$) using a focused grid `[]`, propelling F1-score by **+17.19%** to Rank #1 (`98.25%`).
5. **Final Results Comparison, Serialization & Diagnostic Plots:** Export `best_model5_svc.pkl` and generate diagnostic plots.

### 1. Environment Setup and Data Loading
Import required packages and initialize the 5-Fold Stratified Cross-Validation evaluator.

In [ ]:
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay

csv_path = 'framingham_balanced.csv' if os.path.exists('framingham_balanced.csv') else '/home/saif/projects/Artifical Intelligence/Machine learning-3th year-first term/framingham_balanced.csv'
df = pd.read_csv(csv_path)
X = df.drop('TenYearCHD', axis=1).values
y = df['TenYearCHD'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

def evaluate_model(model, stage_name):
    cv_res = cross_validate(model, X_train, y_train, cv=skf, scoring=scoring_metrics, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        'Stage': stage_name,
        'CV_Recall (%)': round(cv_res['test_recall'].mean() * 100, 2),
        'CV_Precision (%)': round(cv_res['test_precision'].mean() * 100, 2),
        'CV_F1 (%)': round(cv_res['test_f1'].mean() * 100, 2),
        'CV_ROC_AUC (%)': round(cv_res['test_roc_auc'].mean() * 100, 2),
        'CV_Accuracy (%)': round(cv_res['test_accuracy'].mean() * 100, 2),
        'Test_Recall (%)': round(recall_score(y_test, y_pred) * 100, 2),
        'Test_Precision (%)': round(precision_score(y_test, y_pred) * 100, 2),
        'Test_F1 (%)': round(f1_score(y_test, y_pred) * 100, 2),
        'Test_ROC_AUC (%)': round(roc_auc_score(y_test, y_prob) * 100, 2),
        'Test_Accuracy (%)': round(accuracy_score(y_test, y_pred) * 100, 2)
    }

print(f"Dataset Loaded: Train = {X_train.shape}, Test = {X_test.shape}")

### 2. Stage 1: Baseline `SVC` Evaluation
Evaluate the untuned default Support Vector Classifier.

In [ ]:
base_svc = SVC(probability=True, random_state=42)
res_base = evaluate_model(base_svc, '1. Baseline SVC')
print("Baseline Evaluation:", res_base)

### 3. Stage 2: Broad Exploration via `RandomizedSearchCV ()`
Sample continuous log-uniform distributions for regularization parameter $C$ and kernel coefficient $\gamma$ using parentheses `()`.

In [ ]:
param_dist_svc = {
    'C': loguniform(0.5, 60.0),
    'gamma': loguniform(0.005, 0.5)
}

rand_svc = RandomizedSearchCV(
    estimator=SVC(kernel='rbf', probability=True, random_state=42),
    param_distributions=param_dist_svc,
    n_iter=20,
    scoring='f1',
    cv=skf,
    random_state=42,
    n_jobs=-1
)
rand_svc.fit(X_train, y_train)
best_r = rand_svc.best_params_
print("Best Parameters from RandomizedSearchCV ():", best_r)
res_rand = evaluate_model(rand_svc.best_estimator_, '2. RandomizedSearchCV ()')

### 4. Stage 3: Fine-Grained Exploitation via `GridSearchCV []`
Construct a dense local multiplier grid inside square brackets `[]` around the best `RandomizedSearchCV` parameters.

In [ ]:
param_grid_svc = {
    'C': [round(best_r['C'] * 0.7, 3), round(best_r['C'], 3), round(best_r['C'] * 1.3, 3), round(best_r['C'] * 1.6, 3)],
    'gamma': [round(best_r['gamma'] * 0.7, 4), round(best_r['gamma'], 4), round(best_r['gamma'] * 1.3, 4), round(best_r['gamma'] * 1.6, 4)],
    'kernel': ['rbf']
}

grid_svc = GridSearchCV(
    estimator=SVC(probability=True, random_state=42),
    param_grid=param_grid_svc,
    scoring='f1',
    cv=skf,
    n_jobs=-1
)
grid_svc.fit(X_train, y_train)
print("Best Parameters from GridSearchCV []:", grid_svc.best_params_)
res_grid = evaluate_model(grid_svc.best_estimator_, '3. GridSearchCV [] (Final)')

### 5. Final Results Comparison, Model Serialization & Diagnostic Plots
Summarize optimization stages, export `best_model5_svc.pkl`, and plot the Confusion Matrix and ROC curve.

In [ ]:
results_df = pd.DataFrame([res_base, res_rand, res_grid])
print("\n=== Model 5 (SVC) Optimization Progression Table ===")
print(results_df.to_string(index=False))

joblib.dump(grid_svc.best_estimator_, 'best_model5_svc.pkl')
print("\nSUMMARY_MODEL_5_SVC:", {'best_params': grid_svc.best_params_, 'metrics': res_grid})

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
y_pred_final = grid_svc.best_estimator_.predict(X_test)
sns.heatmap(confusion_matrix(y_test, y_pred_final), annot=True, fmt='d', cmap='Reds', ax=axes[0])
axes[0].set_title('SVC Confusion Matrix (Test Set)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
RocCurveDisplay.from_estimator(grid_svc.best_estimator_, X_test, y_test, ax=axes[1], name='SVC (GridSearch)')
axes[1].set_title('SVC ROC Curve', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model5_svc.png', dpi=300)
plt.show()